# 04 — Recover bits from an on-off keyed (OOK) carrier

**Goal:** detect bits from a noisy, carrier-modulated signal, rather than a simple high/low voltage level.

Each bit occupies **24 samples**. An OOK `1` has a large sine-wave carrier; an OOK `0` has little/no carrier, but noise remains.

**Why the UART threshold method fails:** the carrier oscillates above and below zero, so even a valid `1` can have a mean close to zero. Instead, compute signal **power** (`mean(x*x)`) per bit.

In [ ]:
import csv
with open('../data/ook_capture.csv', newline='') as f:
    samples = [float(row['amplitude']) for row in csv.DictReader(f)]

samples_per_bit = 24
print('Loaded', len(samples), 'samples =', len(samples) // samples_per_bit, 'symbols')
print('Carrier samples:', [round(x, 2) for x in samples[:12]])

## Exercise 1: compute signal power

Implement mean-squared power: `P = sum(x*x for x in window) / len(window)`.

Python: `x*x for x in window` is a **generator expression**, which produces values as `sum()` consumes them. If that's unfamiliar, use a conventional loop and accumulator.

In [ ]:
def mean_power(window):
    # TODO: return mean of squared values
    raise NotImplementedError

assert abs(mean_power([1, -1, 1, -1]) - 1.0) < 1e-9
assert mean_power([0, 0]) == 0
print('Power test passed')

## Exercise 2: classify each symbol with its energy

For each 24-sample symbol, calculate mean power and decide 1 if power exceeds `0.18`, otherwise 0. Concatenate bits with `''.join(...)` only after you're confident the output is correct.

In [ ]:
def decode_ook(values, samples_per_bit=24, threshold=0.18):
    bits = []
    # TODO: slice each symbol; calculate mean power; append bit
    raise NotImplementedError

expected = '101100101011'
actual = ''.join(str(x) for x in decode_ook(samples))
assert actual == expected
print('Decoded symbols:', actual)

## Exercise 3: inspect raw amplitude vs energy

A carrier's raw amplitude alternates sign. Squaring samples makes all energy contributions positive. You could average squared samples across a window for a smooth **power envelope**, then threshold it.

This is an envelope/energy detector suitable for this OOK example. For FSK, frequency correlation is a better detector; the bit depends on **which tone**, not overall power.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('Optional: pip install matplotlib to display plots')
else:
    powers = [mean_power(samples[i:i+samples_per_bit])
              for i in range(0, len(samples), samples_per_bit)]
    fig, axes = plt.subplots(2, 1, figsize=(10, 5))
    axes[0].plot(samples)
    axes[0].set_title('Noisy OOK carrier')
    axes[0].set_xlabel('Sample index')
    axes[1].plot(powers, marker='o')
    axes[1].axhline(0.18, color='gray', linestyle='--')
    axes[1].set_title('Average power per symbol (threshold at 0.18)')
    axes[1].set_xlabel('Symbol index')
    plt.tight_layout()
    plt.show()

## Bonus (advanced): FSK uses frequency, not amplitude

If 0 and 1 use different sine-wave frequencies, each bit may have similar total energy. One straightforward detector correlates a window against sine and cosine at each candidate frequency:

`I = sum(x[n]*cos(2*pi*f*n))`, `Q = sum(x[n]*sin(2*pi*f*n))`, `score = I*I + Q*Q`.

Whichever candidate frequency gets the larger score is the estimated transmitted tone. This is more robust to unknown carrier phase than counting zero crossings. We can work through the Python implementation together after you've finished the OOK challenge.